In [ ]:
import time
from collections import deque
from itertools import permutations


def in_mt(mt):
    mt = tuple(mt)
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i * 3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res


def get_successors(state):
    state = tuple(state)
    pos = state.index(0)
    r, c = pos // 3, pos % 3

    successors = []

    def swap_tuple(s, i, j):
        new_s = list(s)
        new_s[i], new_s[j] = new_s[j], new_s[i]
        return tuple(new_s)

    if c > 0:
        successors.append(("Trái", swap_tuple(state, pos, pos - 1)))

    if c < 2:
        successors.append(("Phải", swap_tuple(state, pos, pos + 1)))

    if r > 0:
        successors.append(("Lên", swap_tuple(state, pos, pos - 3)))

    if r < 2:
        successors.append(("Xuống", swap_tuple(state, pos, pos + 3)))

    return successors


def is_successor(s1, s2):
    s1 = tuple(s1)
    s2 = tuple(s2)

    for action, child in get_successors(s1):
        if child == s2:
            return True

    return False


def get_action(s1, s2):
    s1 = tuple(s1)
    s2 = tuple(s2)

    for action, child in get_successors(s1):
        if child == s2:
            return action

    return None


def constraint_ok(xi, vi, xj, vj):
    i = int(xi[1:])
    j = int(xj[1:])

    # Xi -> Xj
    if j == i + 1:
        return is_successor(vi, vj)

    # Xj -> Xi
    if j == i - 1:
        return is_successor(vj, vi)

    return False


def revise(domains, xi, xj, stats):
    revised = False

    for vi in list(domains[xi]):
        has_support = False

        for vj in domains[xj]:
            if constraint_ok(xi, vi, xj, vj):
                has_support = True
                break

        if not has_support:
            domains[xi].remove(vi)
            revised = True
            stats["removed"] += 1

    if revised:
        stats["revise_count"] += 1

    return revised


def ac3(domains, k):
    queue = deque()
    stats = {
        "revise_count": 0,
        "removed": 0,
        "arcs_checked": 0
    }

    # Tạo các cung:
    # (X0, X1), (X1, X0), (X1, X2), (X2, X1), ...
    for i in range(k):
        xi = f"X{i}"
        xj = f"X{i + 1}"
        queue.append((xi, xj))
        queue.append((xj, xi))

    neighbors = {}

    for i in range(k + 1):
        var = f"X{i}"
        neighbors[var] = []

        if i > 0:
            neighbors[var].append(f"X{i - 1}")

        if i < k:
            neighbors[var].append(f"X{i + 1}")

    while queue:
        xi, xj = queue.popleft()
        stats["arcs_checked"] += 1

        if revise(domains, xi, xj, stats):
            if len(domains[xi]) == 0:
                return False, stats

            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True, stats


def create_domains_ac3_only(start_state, goal_state, k):
    start_state = tuple(start_state)
    goal_state = tuple(goal_state)

    all_states = set(permutations(range(9)))

    domains = {}

    for i in range(k + 1):
        var = f"X{i}"

        if i == 0:
            domains[var] = {start_state}

        elif i == k:
            domains[var] = {goal_state}

        else:
            # Miền ban đầu của Xi là toàn bộ trạng thái 8-puzzle có thể có.
            domains[var] = set(all_states)

    return domains


def read_solution_if_singleton(domains, k):
    # Chỉ đọc nghiệm nếu AC-3 đã lọc mỗi biến còn đúng 1 giá trị.
    for i in range(k + 1):
        var = f"X{i}"
        if len(domains[var]) != 1:
            return None

    states = []

    for i in range(k + 1):
        var = f"X{i}"
        states.append(next(iter(domains[var])))

    path = []

    for i in range(k):
        s1 = states[i]
        s2 = states[i + 1]

        if not is_successor(s1, s2):
            return None

        action = get_action(s1, s2)
        path.append((action, s2))

    return path


def ac3_only(start_state, goal_state, k):
    start_time = time.time()

    domains = create_domains_ac3_only(start_state, goal_state, k)

    initial_domain_size = sum(len(domains[var]) for var in domains)

    result, stats = ac3(domains, k)

    final_domain_size = sum(len(domains[var]) for var in domains)

    end_time = time.time()

    stats["initial_domain_size"] = initial_domain_size
    stats["final_domain_size"] = final_domain_size
    stats["time"] = end_time - start_time

    if not result:
        return None, domains, stats

    path = read_solution_if_singleton(domains, k)

    return path, domains, stats